# Solution two : data augmentation + Train + log to MLFlow

# 0.Library

In [ ]:
# general
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import re
import importlib

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_validate
from sklearn.pipeline import Pipeline

# MLFlow
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# Paths
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp
from features import general_func as gf
import constants_data as cd
from features import model_utils as mu

# Reload
importlib.reload(du)
importlib.reload(dp)
importlib.reload(gf)
importlib.reload(cd)
importlib.reload(mu)

# 1. Paths and Constants

In [ ]:
# Read dataframe
parent_folder = Path("../..") # go 2 folder up= "../.."
df_path = parent_folder / "data" / "produced_csv" / "2.cleaned_features_13_patients.csv"
 
df = pd.read_csv(df_path)

df.head()

# 2. Data visualization of augmented data with 13 real patients

In [ ]:
plt.hist(df['MoCA_Score'], bins=10)
plt.title("MoCA Score Distribution")
plt.xlabel("Score")
plt.ylabel("Count")
plt.show()

# 3. Data Splitting

In [ ]:
y = df['MoCA_Score']
df = df.drop(columns=['MoCA_Score'])
X = df

# 4. Train and Test

In [ ]:
cd.all_experiments_sol_1, cd.all_experiments_sol_2

In [ ]:
# experiment list
experiments_list = cd.all_experiments_sol_2

for exp_name, models_list in experiments_list:
    
    print(f"======== Experiment: {exp_name} ===========")
    mlflow.set_experiment(f"MoCA_Regression_data_augmentation_{exp_name}")

    for model in models_list:
        print(f"================ Run: {exp_name}_{model.__name__} =================")
        with mlflow.start_run(run_name=f"{exp_name}_{model.__name__}"):

            pipe = mu.make_model_pipeline(
                mu.make_feature_pipeline(exp_name, k_value),
                model()
            )

            scores = cross_validate(
                pipe,
                X,
                y,
                cv=5,
                scoring={
                    "r2": "r2",
                    "mae": "neg_mean_absolute_error",
                    "mse": "neg_mean_squared_error",
                    "rmse": "neg_root_mean_squared_error"
                },
                return_train_score=True
            )

            # compute metrics of test set
            r2_test   = scores["test_r2"].mean()
            mae_test  = -scores["test_mae"].mean()
            mse_test  = -scores["test_mse"].mean()
            rmse_test = -scores["test_rmse"].mean()

            # compute metrics of train set
            r2_train   = scores["train_r2"].mean()
            mae_train  = -scores["train_mae"].mean()
            mse_train  = -scores["train_mse"].mean()
            rmse_train = -scores["train_rmse"].mean()

            # print
            print(f"Model: {model.__name__}")
            print(f"R² of test: {mse_test:.3f}")

            # MLflow logging
            mlflow.log_param("experiment", exp_name)
            mlflow.log_param("model", model.__name__)

            mlflow.log_metric("r2_test", r2_test)
            mlflow.log_metric("mae_test", mae_test)
            mlflow.log_metric("mse_test", mse_test)
            mlflow.log_metric("rmse_test", rmse_test)

            mlflow.log_metric("r2_test", r2_train)
            mlflow.log_metric("mae_test", mae_train)
            mlflow.log_metric("mse_test", mse_train)
            mlflow.log_metric("rmse_test", rmse_train)

            mlflow.log_metric("gap", abs(mse_test - mse_train))

            # log model
            mlflow.sklearn.log_model(pipe, name="model", skops_trusted_types=["xgboost.sklearn.XGBRegressor", "sklearn.feature_selection._univariate_selection.f_regression"])


## 5.0 Split train/test and Scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)

In [ ]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5.1 Gaussian noise

In [ ]:
X_train_aug , y_train_aug = du.guassian_noise_pipeline(loc=0, scale=1, 
                                                       X_train_scaled= X_train_scaled, y_train= y_train, 
                                                       original_cols= X_train.columns)

In [ ]:
X_train_aug.shape , y_train_aug.shape

## 5.2 SMOTER

In [ ]:
import numpy as np

Q1 = np.quantile(y_train, 0.25)
Q2 = np.quantile(y_train, 0.50)  # median
Q3 = np.quantile(y_train, 0.75)

print("Q1:", Q1)
print("Q2:", Q2)
print("Q3:", Q3)

IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print("lower_fence:",lower_fence)
print("upper_fence:", upper_fence)